# iMAGE: Real-Noise Image Denoising
## Data validation, fine-tuning and test evaluation of Restormer, DnCNN and NAFNet

**CCS250 Computer Vision, Final Project.** This single notebook replaces the three development notebooks (`01_prepare_5000_images`, `02_finetune_models`, `03_evaluate_and_deploy`). Run it top to bottom (**Run All**).

| Stage | What happens | Output |
|---|---|---|
| 0 | Paste the dataset path, locate the project, check the GPU | resolved paths |
| 1 | Build `splits.csv` only if it is missing (SIDD 3,500 + PolyU 1,500) | `data/splits.csv` |
| 2 | **Validate every file and the split** (counts, files, pair alignment, scene leakage, loader) | `validation_report.json` |
| 3 | Fetch pretrained weights, smoke-test all three models | pass/fail per model |
| 4 | Fine-tune DnCNN, NAFNet and Restormer on the train split (validated on the val split) | `*_finetuned.pth` |
| 5 | Training curves | PNG graphs and CSV logs |
| 6 | Evaluate on the untouched 500-image **test** split (noisy input vs pretrained vs fine-tuned) | tables, charts, sample images |
| 7 | Export everything for submission | `submission_output/` and a `.zip` |

**The test split is used only in Stage 6, after all training is finished.**

**Project notes carried over from the original notebooks:** the project owner reports instructor approval for the three-pretrained option and NAFNet. The published Restormer and NAFNet checkpoints were trained on SIDD training images, which overlap the source collection used here, so SIDD and PolyU are always reported separately. The published DnCNN checkpoint was trained on Gaussian noise.

## Stage 0. Configuration (the only cell you need to edit)

In [ ]:
# =====================================================================
#  EDIT THIS CELL ONLY, THEN RUN ALL
# =====================================================================

# 1) PASTE THE PATH TO YOUR DATASET. Any ONE of these works:
#      - the project folder         (the folder that contains app/ and data/)
#      - the dataset folder         (.../data, the folder that contains splits.csv)
#      - the manifest file itself   (.../data/splits.csv)
#    Windows: keep the r"..." prefix. Quotes copied by "Copy as path" are removed automatically.
#    Leave "" to auto-detect (current folder, or Colab: MyDrive/CCS250_CompVis_Final).
DATASET_PATH = r""

# 2) Training settings (defaults reproduce the original notebook 02 "full" run)
MODELS         = ['dncnn', 'nafnet', 'restormer']
EPOCHS         = 5
LEARNING_RATE  = 1e-5
PATCH          = 128                                   # training crop size
BATCH_SIZE     = {'dncnn': 2, 'nafnet': 1, 'restormer': 1}
VALIDATION_LIMIT = 500                                 # validation images used per epoch
SEED           = 42

# 3) Evaluation settings (match original notebook 03)
EVAL_PATCH     = 64                                    # test crop size (center crop)
VIS_CROP       = 128                                   # crop size for the sample-image figure
EVAL_PRETRAINED_BASELINE = True                        # also score the published checkpoints for comparison

# 4) Behaviour switches
RUN_SMOKE_PREFLIGHT   = True    # 5-step trial of every model BEFORE the long training run
FORCE_RETRAIN         = False   # False = skip a model already trained with identical settings (safe to resume)
REQUIRE_CUDA          = True    # stop early if no GPU (full training on CPU is impractical)
BUILD_MANIFEST_IF_MISSING = True
FULL_DECODE_CHECK     = True    # fully decode every image during validation (slower, catches corrupt files)
SANITY_SAMPLE         = 60      # pairs used for the alignment / noise-level sanity check
INSTALL_REQUIREMENTS  = 'auto'  # 'auto' = only on Colab; True / False to force
EXPORT_ZIP            = True
DOWNLOAD_ZIP_IN_COLAB = True

# 5) What the manifest must contain (change only if your split differs)
EXPECTED_SPLITS  = {'train': 4000, 'val': 500, 'test': 500}
EXPECTED_SOURCES = {'SIDD': 3500, 'PolyU': 1500}

In [ ]:
# ---------- Locate the project and the dataset ----------
from pathlib import Path
import os, sys, json, csv, hashlib, time, shutil, platform, random, subprocess, datetime, gc

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

def as_path(text):
    return Path(str(text).strip().strip('"').strip("'")).expanduser()

def is_project(p):
    return (p / 'app').is_dir() and ((p / 'app' / 'experiments.py').is_file() or (p / 'app' / 'experiments').is_dir())

user_path = as_path(DATASET_PATH) if str(DATASET_PATH).strip() else None
if user_path is not None and not user_path.exists():
    raise FileNotFoundError(f'DATASET_PATH does not exist: {user_path}')

hints = []
if user_path is not None: hints.append(user_path.resolve())
hints += [Path.cwd().resolve(), Path('/content/drive/MyDrive/CCS250_CompVis_Final')]
ROOT = None
for hint in hints:
    for cand in [hint, *hint.parents]:
        if is_project(cand):
            ROOT = cand.resolve(); break
    if ROOT: break
if ROOT is None:
    raise FileNotFoundError('Could not find the project (a folder containing app/experiments.py). '
                            'Paste the project folder into DATASET_PATH, or open the project folder in VS Code / Drive.')

def resolve_data_dir(p):
    if p.is_file():
        return p.parent.resolve() if p.name == 'splits.csv' else None
    for cand in (p, p / 'data'):
        if (cand / 'splits.csv').is_file() or (cand / 'raw').is_dir() or (cand / 'inventory').is_dir():
            return cand.resolve()
    return None

APP_DATA = ROOT / 'data'
if user_path is None or user_path.resolve() == ROOT:
    DATA = APP_DATA
else:
    DATA = resolve_data_dir(user_path.resolve())
    if DATA is None:
        raise FileNotFoundError(f'No splits.csv, raw/ or inventory/ found at or under: {user_path}')

# The project code (app/*.py) reads the dataset from <project>/data, so the dataset must be reachable there.
if DATA.resolve() != APP_DATA.resolve():
    if APP_DATA.exists():
        raise RuntimeError(f'{APP_DATA} already exists and is a different folder from {DATA}. '
                           'Point DATASET_PATH at the project folder or at that data/ folder, or rename one of them.')
    try:
        APP_DATA.symlink_to(DATA, target_is_directory=True)
        print(f'Linked {APP_DATA} -> {DATA}')
    except OSError as exc:
        raise RuntimeError(f'Could not link {DATA} into {APP_DATA} ({exc}). Copy or move the dataset to {APP_DATA}.')
DATA = APP_DATA
DATA.mkdir(exist_ok=True)

os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
WEIGHTS, RESULTS = ROOT / 'weights', ROOT / 'results'
WEIGHTS.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)
OUT = ROOT / 'submission_output'
for sub in ('weights', 'graphs', 'test_results', 'logs'): (OUT / sub).mkdir(parents=True, exist_ok=True)

print('Project root :', ROOT)
print('Dataset (data):', DATA, '(exists)' if DATA.exists() else '')
print('Manifest     :', DATA / 'splits.csv', '-> found' if (DATA / 'splits.csv').is_file() else '-> NOT found yet (will be built in Stage 1)')
print('Weights dir  :', WEIGHTS)
print('Export dir   :', OUT)

In [ ]:
# ---------- Dependencies, device, reproducibility ----------
if INSTALL_REQUIREMENTS is True or (INSTALL_REQUIREMENTS == 'auto' and IN_COLAB):
    req = ROOT / 'requirements.txt'
    if req.is_file():
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
    else:
        print('requirements.txt not found; skipping install')

import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter, defaultdict

from app.experiments import dataset_counts, setup_device, get_network, fine_tune
from app.data import DenoiseDataset
from app.online import pretrained_path, architecture_path

DEVICE = setup_device()
if REQUIRE_CUDA and DEVICE.type != 'cuda':
    raise RuntimeError('No CUDA GPU detected. In Colab choose Runtime > Change runtime type > GPU, '
                       'or set REQUIRE_CUDA = False to continue on CPU (very slow).')

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

ENV = {'python': platform.python_version(), 'platform': platform.platform(), 'torch': torch.__version__,
       'device': str(DEVICE), 'gpu': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
       'numpy': np.__version__, 'pandas': pd.__version__,
       'run_started': datetime.datetime.now().isoformat(timespec='seconds')}
(OUT / 'logs' / 'environment.json').write_text(json.dumps(ENV, indent=2), encoding='utf8')
print(json.dumps(ENV, indent=2))

## Stage 1. Build the paired-image manifest (only if `data/splits.csv` is missing)

If the manifest already exists it is **never overwritten** (the same fixed split must be shared by all three models). Otherwise it is built from the raw SIDD (`train_sub512/{GT,LQ}`, PNG) and PolyU (`train_sub/{GT,LQ}`, JPG) subsets: 4,000 train / 500 validation / 500 test, whole scenes kept in one split.

In [ ]:
manifest = DATA / 'splits.csv'

# Raw-data locations (only used when the manifest has to be built)
SIDD_BASE  = DATA / 'raw/SIDD/SIDD/train_sub512'
POLYU_BASE = DATA / 'raw/PolyU/PolyU-Real-World-Noisy-Images-Dataset/train_sub'

if manifest.is_file():
    print('Using existing manifest (not modified):', manifest)
elif not BUILD_MANIFEST_IF_MISSING:
    raise FileNotFoundError(f'{manifest} is missing and BUILD_MANIFEST_IF_MISSING is False')
else:
    INVENTORY = DATA / 'inventory'; INVENTORY.mkdir(parents=True, exist_ok=True)
    for name, base in (('SIDD', SIDD_BASE), ('PolyU', POLYU_BASE)):
        for part in ('GT', 'LQ'):
            if not (base / part).is_dir():
                raise FileNotFoundError(f'{name} {part} folder not found: {base / part}. Adjust SIDD_BASE / POLYU_BASE above.')

    def write_inventory(name, base, suffix):
        output = INVENTORY / f'{name}.csv'
        if output.exists():
            print('Using existing inventory:', output); return output
        gt, lq = base / 'GT', base / 'LQ'
        clean_files = sorted(gt.glob(f'*{suffix}'))
        if not clean_files: raise ValueError(f'No {suffix} clean images in {gt}')
        records = []
        for clean in clean_files:
            noisy = lq / clean.name
            if not noisy.is_file(): raise FileNotFoundError(f'Missing noisy mate: {noisy}')
            stem = clean.stem.rsplit('_s', 1)[0]
            group = stem.split('_')[0] if name == 'SIDD' else stem
            if not group: raise ValueError(f'Cannot identify scene for {clean.name}')
            with Image.open(clean) as a, Image.open(noisy) as b:
                if a.size != b.size: raise ValueError(f'Dimension mismatch: {clean.name}')
            records.append({'noisy_path': noisy.relative_to(DATA).as_posix(),
                            'clean_path': clean.relative_to(DATA).as_posix(), 'group_id': group})
        with output.open('w', newline='', encoding='utf8') as fh:
            w = csv.DictWriter(fh, fieldnames=['noisy_path', 'clean_path', 'group_id'])
            w.writeheader(); w.writerows(records)
        print(name, len(records), 'pairs,', len({r['group_id'] for r in records}), 'scene groups')
        return output

    write_inventory('SIDD', SIDD_BASE, '.png')
    write_inventory('PolyU', POLYU_BASE, '.jpg')
    from scripts.prepare_mixed_dataset import prepare
    manifest = Path(prepare(seed=42))
    print('Manifest created:', manifest)

print('Counts:', dataset_counts(manifest))

## Stage 2. Validate the dataset

Every check prints `PASS`, `WARN` or `FAIL`. **Any FAIL stops the notebook before training starts**, so a broken dataset cannot waste a long GPU run. Checks: manifest columns, split/source counts, duplicate entries, scene leakage across splits, every image exists and decodes, noisy/clean sizes match, crops fit, spatial alignment, noise-level sanity, and the project's own data loader.

In [ ]:
errors, warnings_ = [], []
def check(ok, pass_msg, fail_msg, warn_only=False):
    if ok:
        print(f'  [PASS] {pass_msg}')
    else:
        (warnings_ if warn_only else errors).append(fail_msg)
        print(f"  [{'WARN' if warn_only else 'FAIL'}] {fail_msg}")

def resolve_path(rel):
    rel = str(rel).replace('\\', '/')
    p = Path(rel)
    if p.is_absolute():
        return p if p.is_file() else None
    for base in (DATA, ROOT):
        if (base / rel).is_file(): return base / rel
    return None

manifest_sha256 = hashlib.sha256(manifest.read_bytes()).hexdigest()
with manifest.open(newline='', encoding='utf8') as fh:
    reader = csv.DictReader(fh)
    columns = reader.fieldnames or []
    rows = list(reader)
print('Manifest :', manifest)
print('SHA-256  :', manifest_sha256)
print('Rows     :', len(rows))

print('\n[1] Manifest structure')
required = {'noisy_path', 'clean_path', 'source', 'split'}
check(required <= set(columns), f'required columns present {sorted(required)}',
      f'missing columns {sorted(required - set(columns))}; found {columns}')
if errors: raise RuntimeError('Manifest has the wrong structure: ' + '; '.join(errors))
check({r['split'] for r in rows} <= set(EXPECTED_SPLITS), 'split labels are train / val / test only',
      f"unexpected split labels: {sorted({r['split'] for r in rows} - set(EXPECTED_SPLITS))}")

print('\n[2] Counts')
split_counts, source_counts = Counter(r['split'] for r in rows), Counter(r['source'] for r in rows)
check(dict(split_counts) == EXPECTED_SPLITS, f'splits = {dict(split_counts)}',
      f'splits {dict(split_counts)} differ from expected {EXPECTED_SPLITS}')
check(dict(source_counts) == EXPECTED_SOURCES, f'sources = {dict(source_counts)}',
      f'sources {dict(source_counts)} differ from expected {EXPECTED_SOURCES}')
check(dict(dataset_counts(manifest)) == EXPECTED_SPLITS, 'app.experiments.dataset_counts agrees',
      f'dataset_counts returned {dataset_counts(manifest)}')
print(pd.crosstab(pd.Series([r['source'] for r in rows], name='source'),
                  pd.Series([r['split'] for r in rows], name='split')).to_string())

print('\n[3] Duplicates and scene leakage')
dups = [p for p, c in Counter(r['noisy_path'] for r in rows).items() if c > 1]
check(not dups, 'no duplicate image pairs', f'{len(dups)} duplicated noisy paths, e.g. {dups[:3]}')
if 'group_id' in columns:
    group_splits = defaultdict(set)
    for r in rows: group_splits[(r['source'], r['group_id'])].add(r['split'])
    leaked = {g: sorted(s) for g, s in group_splits.items() if len(s) > 1}
    check(not leaked, f'{len(group_splits)} scene groups, none shared between splits',
          f'{len(leaked)} scene groups appear in more than one split, e.g. {list(leaked.items())[:3]}')
else:
    check(False, '', 'manifest has no group_id column, so scene leakage cannot be verified', warn_only=True)

print('\n[4] Files exist, decode, and noisy/clean sizes match')
missing, unreadable, mismatched, too_small = [], [], [], []
t0 = time.time()
for i, r in enumerate(rows):
    pn, pc = resolve_path(r['noisy_path']), resolve_path(r['clean_path'])
    if pn is None or pc is None:
        missing.append((r['noisy_path'] if pn is None else r['clean_path'])); continue
    try:
        dims = []
        for p in (pn, pc):
            with Image.open(p) as im:
                if FULL_DECODE_CHECK: im.load()
                dims.append(im.size)
    except Exception as exc:
        unreadable.append((pn.name, repr(exc))); continue
    if dims[0] != dims[1]: mismatched.append((pn.name, dims))
    if min(dims[0]) < PATCH: too_small.append((pn.name, dims[0]))
    if (i + 1) % 1000 == 0: print(f'    checked {i + 1}/{len(rows)} pairs ({time.time() - t0:.0f}s)')
check(not missing, f'all {2 * len(rows)} image files found', f'{len(missing)} files missing, e.g. {missing[:3]}')
check(not unreadable, 'every image decodes' if FULL_DECODE_CHECK else 'every image header opens',
      f'{len(unreadable)} unreadable images, e.g. {unreadable[:3]}')
check(not mismatched, 'noisy and clean dimensions match for every pair',
      f'{len(mismatched)} pairs with different noisy/clean sizes, e.g. {mismatched[:3]}')
check(not too_small, f'every image is at least {PATCH}x{PATCH}',
      f'{len(too_small)} images smaller than the {PATCH}px training crop, e.g. {too_small[:3]}')

print('\n[5] Alignment and noise-level sanity (random sample)')
def to_float(path, side=256):
    with Image.open(path) as im:
        im = im.convert('RGB'); w, h = im.size; s = min(side, w, h)
        l, t = (w - s) // 2, (h - s) // 2
        return np.asarray(im.crop((l, t, l + s, t + s)), dtype=np.float32) / 255.0

rng = random.Random(SEED)
pool = [r for r in rows if resolve_path(r['noisy_path']) and resolve_path(r['clean_path'])]
sample_rows = rng.sample(pool, min(SANITY_SAMPLE, len(pool)))
psnr_by_source, identical, shifted = defaultdict(list), [], []
for r in sample_rows:
    n, c = to_float(resolve_path(r['noisy_path'])), to_float(resolve_path(r['clean_path']))
    if n.shape != c.shape: continue
    m = 4; h, w, _ = n.shape
    core = n[m:h - m, m:w - m]
    def mse_at(dy, dx): return float(((core - c[m + dy:h - m + dy, m + dx:w - m + dx]) ** 2).mean())
    base = mse_at(0, 0)
    best = min((mse_at(dy, dx), (dy, dx)) for dy in range(-2, 3) for dx in range(-2, 3))
    if base < 1e-10: identical.append(r['noisy_path'])
    elif best[1] != (0, 0) and best[0] < 0.9 * base: shifted.append((r['noisy_path'], best[1]))
    psnr_by_source[r['source']].append(10 * np.log10(1.0 / max(base, 1e-10)))
for s, v in psnr_by_source.items():
    print(f'    {s}: noisy-vs-clean PSNR over {len(v)} sampled pairs: mean {np.mean(v):.1f} dB (min {np.min(v):.1f}, max {np.max(v):.1f})')
check(not identical, 'no noisy image is identical to its clean mate', f'{len(identical)} sampled pairs have noisy == clean, e.g. {identical[:3]}')
check(not shifted, 'no sign of spatial misalignment (+/-2 px shift test)',
      f'{len(shifted)} sampled pairs match better after a pixel shift, so they may be misaligned: {shifted[:3]}', warn_only=True)
all_psnr = [x for v in psnr_by_source.values() for x in v]
check(all(10 <= x <= 45 for x in all_psnr), 'noisy/clean PSNR is in a plausible 10-45 dB range',
      'some sampled pairs have an implausible noisy/clean PSNR; inspect them visually', warn_only=True)

print('\n[6] Project data loader (app.data.DenoiseDataset)')
for split, patch in (('train', PATCH), ('val', PATCH), ('test', EVAL_PATCH)):
    try:
        ds = DenoiseDataset(manifest, split, patch=patch)
        x, y = ds[0]
        good = (len(ds) == EXPECTED_SPLITS[split] and tuple(x.shape) == tuple(y.shape) == (3, patch, patch)
                and bool(torch.isfinite(x).all()) and bool(torch.isfinite(y).all()))
        check(good, f'{split}: {len(ds)} samples, pair shape {tuple(x.shape)}, finite values',
              f'{split}: loader problem (len={len(ds)}, shapes={tuple(x.shape)}/{tuple(y.shape)})')
        check(0.0 <= float(x.min()) and float(x.max()) <= 1.0 and 0.0 <= float(y.min()) and float(y.max()) <= 1.0,
              f'{split}: values within [0, 1]', f'{split}: tensor values outside [0, 1]; check normalisation', warn_only=True)
    except Exception as exc:
        errors.append(f'{split}: loader raised {exc!r}'); print(f'  [FAIL] {split}: loader raised {exc!r}')

report = {'manifest': str(manifest), 'manifest_sha256': manifest_sha256, 'rows': len(rows),
          'split_counts': dict(split_counts), 'source_counts': dict(source_counts),
          'full_decode_check': FULL_DECODE_CHECK, 'errors': errors, 'warnings': warnings_,
          'passed': not errors, 'checked_at': datetime.datetime.now().isoformat(timespec='seconds')}
(OUT / 'logs' / 'validation_report.json').write_text(json.dumps(report, indent=2), encoding='utf8')
print('\n' + '=' * 60)
print(f'VALIDATION {"PASSED" if not errors else "FAILED"}: {len(errors)} failure(s), {len(warnings_)} warning(s)')
if errors: raise RuntimeError('Dataset validation failed; fix the items marked FAIL above before training.')

## Stage 3. Fetch pretrained weights and smoke-test every model

Pretrained checkpoints are downloaded on first use and cached. Restormer and NAFNet import third-party architecture source. Each model gets a forward-pass test now, so a download or compatibility problem stops the run here instead of hours into training. If a remote checkpoint is unavailable, fix the source; the notebook never substitutes a randomly initialised model.

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(chunk), b''): h.update(block)
    return h.hexdigest()

def free_gpu():
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

probe_noisy, probe_clean = DenoiseDataset(manifest, 'train', patch=PATCH)[0]
PRETRAINED = {}
for m in MODELS:
    if m in ('restormer', 'nafnet'):
        print(f'{m} architecture:', architecture_path(m))
    ckpt = Path(pretrained_path(m))
    PRETRAINED[m] = ckpt
    model, _ = get_network(m, device=DEVICE, checkpoint=ckpt)
    model.eval()
    with torch.inference_mode():
        x = probe_noisy.unsqueeze(0).to(DEVICE)
        pred = model(x)
    assert pred.shape == x.shape and torch.isfinite(pred).all(), f'{m}: forward pass failed'
    print(f'[PASS] {m}: pretrained checkpoint {ckpt.name}, forward {tuple(pred.shape)} on {pred.device}')
    del model, pred, x; free_gpu()

## Stage 4. Fine-tune

Bookkeeping first: a model that already finished a full run with **identical settings and manifest** is skipped (so a disconnected Colab session can simply be re-run). Any existing `weights/{model}_finetuned.pth` that is about to be replaced is backed up to `weights/backup_before_submission_run/`.

Then every model gets a 5-step smoke run (optional), followed by the full run: **5 epochs, L1 loss, learning rate 1e-5, validation on the 500 validation images after every epoch**. The test split is not touched.

In [ ]:
def ft_path(m):    return WEIGHTS / f'{m}_finetuned.pth'
def log_path(m):   return RESULTS / f'{m}_training.csv'
def meta_path(m):  return RESULTS / f'{m}_run_meta.json'

def run_config(m):
    return {'model': m, 'epochs': EPOCHS, 'lr': LEARNING_RATE, 'batch_size': BATCH_SIZE[m], 'patch': PATCH,
            'validation_limit': VALIDATION_LIMIT, 'seed': SEED, 'manifest_sha256': manifest_sha256}

def is_trained(m):
    if not (ft_path(m).is_file() and log_path(m).is_file() and meta_path(m).is_file()): return False
    meta = json.loads(meta_path(m).read_text(encoding='utf8'))
    return (meta.get('mode') == 'full' and meta.get('config') == run_config(m)
            and meta.get('checkpoint_sha256') == sha256_file(ft_path(m)))

TO_TRAIN = [m for m in MODELS if FORCE_RETRAIN or not is_trained(m)]
SKIPPED  = [m for m in MODELS if m not in TO_TRAIN]
print('Will train :', TO_TRAIN or 'nothing')
print('Already done (identical settings, skipped):', SKIPPED or 'none')

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
for m in TO_TRAIN:
    if ft_path(m).is_file():
        backup = WEIGHTS / 'backup_before_submission_run'; backup.mkdir(exist_ok=True)
        shutil.copy2(ft_path(m), backup / f'{m}_finetuned_{stamp}.pth')
        print(f'Backed up existing {ft_path(m).name} -> {backup.name}/')

In [ ]:
# ---------- Smoke pre-flight: 1 epoch, 5 steps, 10 validation images per model ----------
if RUN_SMOKE_PREFLIGHT and TO_TRAIN:
    for m in TO_TRAIN:
        print(f'\n=== SMOKE TEST: {m} ===')
        fine_tune(m, epochs=1, batch_size=BATCH_SIZE[m], patch=PATCH, lr=LEARNING_RATE,
                  validation_limit=10, max_train_steps=5, seed=SEED, checkpoint=PRETRAINED[m])
        assert ft_path(m).is_file() and log_path(m).is_file(), f'{m}: smoke run did not write its outputs'
        print(f'[PASS] {m}: smoke run wrote {ft_path(m).name} and {log_path(m).name}')
        free_gpu()
else:
    print('Smoke pre-flight skipped.')

In [ ]:
# ---------- Full fine-tuning ----------
TRAIN_SECONDS = {}
for m in TO_TRAIN:
    print(f'\n=== FULL FINE-TUNING: {m}  ({EPOCHS} epochs, batch {BATCH_SIZE[m]}, patch {PATCH}, lr {LEARNING_RATE}) ===')
    seed_everything(SEED)
    t0 = time.time()
    fine_tune(m, epochs=EPOCHS, batch_size=BATCH_SIZE[m], patch=PATCH, lr=LEARNING_RATE,
              validation_limit=VALIDATION_LIMIT, max_train_steps=0, seed=SEED, checkpoint=PRETRAINED[m])
    TRAIN_SECONDS[m] = time.time() - t0
    assert ft_path(m).is_file() and log_path(m).is_file(), f'{m}: training did not write its outputs'
    meta = {'mode': 'full', 'config': run_config(m), 'train_seconds': round(TRAIN_SECONDS[m], 1),
            'checkpoint_sha256': sha256_file(ft_path(m)), 'finished_at': datetime.datetime.now().isoformat(timespec='seconds')}
    meta_path(m).write_text(json.dumps(meta, indent=2), encoding='utf8')
    print(f'Finished {m} in {TRAIN_SECONDS[m] / 60:.1f} min -> {ft_path(m)}')
    free_gpu()
print('\nTraining stage complete.')

In [ ]:
# ---------- Verify the produced .pth files ----------
def load_ckpt(path):
    try:    return torch.load(path, map_location='cpu', weights_only=True)
    except Exception: return torch.load(path, map_location='cpu', weights_only=False)

def tensors_of(obj):
    if isinstance(obj, torch.nn.Module): return obj.state_dict()
    if isinstance(obj, dict):
        for key in ('params', 'params_ema', 'state_dict', 'model_state_dict', 'model'):
            if isinstance(obj.get(key), dict): return obj[key]
        return obj
    return {}

CKPT_INFO = {}
for m in MODELS:
    p = ft_path(m)
    assert p.is_file(), f'{p} is missing'
    tensors = {k: v for k, v in tensors_of(load_ckpt(p)).items() if torch.is_tensor(v)}
    floats = [v for v in tensors.values() if v.is_floating_point()]
    finite = all(bool(torch.isfinite(v).all()) for v in floats)
    assert tensors and finite, f'{p.name}: empty or contains NaN/Inf'
    CKPT_INFO[m] = {'file': p.name, 'size_mb': round(p.stat().st_size / 1e6, 2), 'tensors': len(tensors),
                    'float_tensor_elements_m': round(sum(v.numel() for v in floats) / 1e6, 3), 'sha256': sha256_file(p)}
    print(f"[PASS] {p.name}: {CKPT_INFO[m]['size_mb']} MB, {CKPT_INFO[m]['tensors']} tensors, all finite")

## Stage 5. Training graphs

Left: mean L1 training loss per epoch. Right: PSNR on the validation split per epoch. (Validation only; no test data.)

In [ ]:
curves = {}
summary_rows = []
for m in MODELS:
    log = pd.read_csv(log_path(m))
    missing_cols = {'epoch', 'train_l1', 'validation_psnr_db'} - set(log.columns)
    assert not missing_cols, f'{log_path(m).name} lacks columns {missing_cols}; found {list(log.columns)}'
    curves[m] = log
    print(f'--- {m} ---'); display(log)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(log.epoch, log.train_l1, marker='o'); axes[0].set(xlabel='Epoch', ylabel='L1 loss', title=f'{m}: training loss')
    axes[1].plot(log.epoch, log.validation_psnr_db, marker='o'); axes[1].set(xlabel='Epoch', ylabel='PSNR (dB)', title=f'{m}: validation PSNR')
    for ax in axes: ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(RESULTS / f'{m}_training_curves.png', dpi=150)
    fig.savefig(OUT / 'graphs' / f'{m}_training_curves.png', dpi=150)
    plt.show()
    best = log.loc[log.validation_psnr_db.idxmax()]
    meta = json.loads(meta_path(m).read_text(encoding='utf8')) if meta_path(m).is_file() else {}
    summary_rows.append({'model': m, 'epochs': int(log.epoch.max()), 'first_train_l1': round(float(log.train_l1.iloc[0]), 5),
                         'final_train_l1': round(float(log.train_l1.iloc[-1]), 5),
                         'first_val_psnr': round(float(log.validation_psnr_db.iloc[0]), 3),
                         'final_val_psnr': round(float(log.validation_psnr_db.iloc[-1]), 3),
                         'best_val_psnr': round(float(best.validation_psnr_db), 3), 'best_epoch': int(best.epoch),
                         'train_minutes': round(meta.get('train_seconds', float('nan')) / 60, 1)})
    shutil.copy2(log_path(m), OUT / 'logs' / log_path(m).name)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for m, log in curves.items():
    axes[0].plot(log.epoch, log.train_l1, marker='o', label=m)
    axes[1].plot(log.epoch, log.validation_psnr_db, marker='o', label=m)
axes[0].set(xlabel='Epoch', ylabel='L1 loss', title='Training loss (all models)')
axes[1].set(xlabel='Epoch', ylabel='PSNR (dB)', title='Validation PSNR (all models)')
for ax in axes: ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(OUT / 'graphs' / 'all_models_training_curves.png', dpi=150); plt.show()

training_summary = pd.DataFrame(summary_rows)
training_summary.to_csv(OUT / 'logs' / 'training_summary.csv', index=False)
display(training_summary)

## Stage 6. Evaluate on the untouched test split

All variants are scored on the **same 500 test pairs** (center crops of `EVAL_PATCH` pixels, RGB, values in [0, 1], PSNR/SSIM with data range 1; runtime is forward inference only, batch size 1, per crop). Compared for each network: the **published (pretrained) checkpoint** and the **new fine-tuned checkpoint**, plus the raw noisy input as the baseline. SSIM uses an 11x11 Gaussian window (sigma 1.5), computed here in PyTorch.

In [ ]:
test_rows = [r for r in rows if r['split'] == 'test']
assert len(test_rows) == EXPECTED_SPLITS['test']

def load_center(rel, crop):
    with Image.open(resolve_path(rel)) as im:
        im = im.convert('RGB'); w, h = im.size
        l, t = (w - crop) // 2, (h - crop) // 2
        arr = np.asarray(im.crop((l, t, l + crop, t + crop)), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

noisy_all = torch.stack([load_center(r['noisy_path'], EVAL_PATCH) for r in test_rows])
clean_all = torch.stack([load_center(r['clean_path'], EVAL_PATCH) for r in test_rows])
test_sources = [r['source'] for r in test_rows]

# a few fixed test images for the side-by-side figure: first of each source, padded to 3
vis_idx = [test_sources.index(s) for s in dict.fromkeys(test_sources)]
for i in range(0, len(test_rows), 97):
    if len(vis_idx) >= 3: break
    if i not in vis_idx: vis_idx.append(i)
vis_idx = vis_idx[:3]
vis_noisy = torch.stack([load_center(test_rows[i]['noisy_path'], VIS_CROP) for i in vis_idx])
vis_clean = torch.stack([load_center(test_rows[i]['clean_path'], VIS_CROP) for i in vis_idx])

def psnr_each(a, b):
    mse = ((a - b) ** 2).mean(dim=(1, 2, 3))
    return (10 * torch.log10(1.0 / mse.clamp_min(1e-10))).numpy()

_g = torch.arange(11, dtype=torch.float32) - 5
_g = torch.exp(-(_g ** 2) / (2 * 1.5 ** 2)); _g = _g / _g.sum()
_W = (_g[:, None] * _g[None, :])[None, None].repeat(3, 1, 1, 1)

def ssim_each(a, b):
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    conv = lambda x: F.conv2d(x, _W, groups=3)
    mu_a, mu_b = conv(a), conv(b)
    s_aa, s_bb, s_ab = conv(a * a) - mu_a ** 2, conv(b * b) - mu_b ** 2, conv(a * b) - mu_a * mu_b
    s = ((2 * mu_a * mu_b + C1) * (2 * s_ab + C2)) / ((mu_a ** 2 + mu_b ** 2 + C1) * (s_aa + s_bb + C2))
    return s.mean(dim=(1, 2, 3)).numpy()

@torch.inference_mode()
def evaluate_variant(name, ckpt, keep_vis=False):
    model, _ = get_network(name, device=DEVICE, checkpoint=ckpt)
    model.eval()
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    sync = (lambda: torch.cuda.synchronize()) if DEVICE.type == 'cuda' else (lambda: None)
    for _ in range(3): model(noisy_all[:1].to(DEVICE)); sync()          # warm-up
    outs, ms = [], []
    for i in range(len(noisy_all)):
        x = noisy_all[i:i + 1].to(DEVICE)
        sync(); t0 = time.perf_counter(); y = model(x); sync()
        ms.append((time.perf_counter() - t0) * 1000)
        assert y.shape == x.shape, f'{name}: output shape {tuple(y.shape)} != input {tuple(x.shape)}'
        outs.append(y.float().clamp(0, 1).cpu())
    out = torch.cat(outs)
    vis = model(vis_noisy.to(DEVICE)).float().clamp(0, 1).cpu() if keep_vis else None
    del model; free_gpu()
    df = pd.DataFrame({'source': test_sources, 'psnr': psnr_each(out, clean_all), 'ssim': ssim_each(out, clean_all), 'ms': ms})
    return df, params_m, vis

def summarise(model, variant, df, params_m, ckpt):
    by = df.groupby('source')
    row = {'model': model, 'variant': variant, 'PSNR (dB)': df.psnr.mean(), 'SSIM': df.ssim.mean(),
           'runtime (ms/crop)': df.ms.mean() if params_m is not None else np.nan,
           'parameters (M)': params_m if params_m is not None else np.nan}
    for s in sorted(by.groups): row[f'{s} PSNR'] = by.get_group(s).psnr.mean(); row[f'{s} SSIM'] = by.get_group(s).ssim.mean()
    row['checkpoint'] = Path(ckpt).name if ckpt else ''
    row['checkpoint_sha256'] = sha256_file(ckpt) if ckpt else ''
    return row

baseline = pd.DataFrame({'source': test_sources, 'psnr': psnr_each(noisy_all, clean_all),
                         'ssim': ssim_each(noisy_all, clean_all), 'ms': np.nan})
RESULT_ROWS = [summarise('(none)', 'noisy input', baseline, None, None)]
PER_IMAGE, VIS_OUT = [baseline.assign(model='(none)', variant='noisy input')], {}

for m in MODELS:
    variants = ([('pretrained', PRETRAINED[m])] if EVAL_PRETRAINED_BASELINE else []) + [('fine-tuned', ft_path(m))]
    for variant, ckpt in variants:
        print(f'Evaluating {m} / {variant} on {len(test_rows)} test crops ...')
        df, params_m, vis = evaluate_variant(m, ckpt, keep_vis=(variant == 'fine-tuned'))
        RESULT_ROWS.append(summarise(m, variant, df, params_m, ckpt))
        PER_IMAGE.append(df.assign(model=m, variant=variant))
        if vis is not None: VIS_OUT[m] = vis

results = pd.DataFrame(RESULT_ROWS)
pre = results[results.variant == 'pretrained'].set_index('model')['PSNR (dB)']
results['dPSNR vs pretrained (dB)'] = [results.loc[i, 'PSNR (dB)'] - pre[results.loc[i, 'model']]
                                       if results.loc[i, 'variant'] == 'fine-tuned' and results.loc[i, 'model'] in pre.index else np.nan
                                       for i in results.index]
TEST_DIR = OUT / 'test_results'
shown = [c for c in results.columns if c not in ('checkpoint', 'checkpoint_sha256')]
display(results[shown].round(3))
results.to_csv(TEST_DIR / 'test_results_table.csv', index=False)
pd.concat(PER_IMAGE).to_csv(TEST_DIR / 'test_per_image.csv', index=False)
(TEST_DIR / 'test_summary.json').write_text(json.dumps({
    'manifest_sha256': manifest_sha256, 'number_of_test_images': len(test_rows), 'eval_patch': EVAL_PATCH,
    'psnr_ssim_data_range': 1.0, 'device': str(DEVICE), 'results': results.replace({np.nan: None}).to_dict('records')}, indent=2), encoding='utf8')

In [ ]:
# ---------- Verdict per model ----------
print('Fine-tuned vs pretrained, test split (PSNR in dB):')
for m in MODELS:
    ft = results[(results.model == m) & (results.variant == 'fine-tuned')].iloc[0]
    line = f"  {m:<10} fine-tuned {ft['PSNR (dB)']:.3f} dB, SSIM {ft['SSIM']:.4f}"
    if EVAL_PRETRAINED_BASELINE:
        d = ft['dPSNR vs pretrained (dB)']
        line += f"  | change vs pretrained {d:+.3f} dB -> " + ('IMPROVED' if d > 0 else 'NOT IMPROVED: consider keeping the pretrained weights for this model in the app')
    print(line)
print('\nNote: published Restormer/NAFNet checkpoints saw SIDD training images, so read SIDD and PolyU numbers separately.')

# ---------- Charts ----------
src_cols = [c for c in results.columns if c.endswith(' PSNR') and c != 'PSNR (dB)']
panels = [('PSNR (dB)', 'Overall')] + [(c, c.replace(' PSNR', '') + (' (pretraining overlap)' if c.startswith('SIDD') else '')) for c in src_cols]
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.2), squeeze=False)
width = 0.38
for ax, (col, title) in zip(axes[0], panels):
    xs = np.arange(len(MODELS))
    if EVAL_PRETRAINED_BASELINE:
        pv = [results[(results.model == m) & (results.variant == 'pretrained')][col].iloc[0] for m in MODELS]
        ax.bar(xs - width / 2, pv, width, label='pretrained')
    fv = [results[(results.model == m) & (results.variant == 'fine-tuned')][col].iloc[0] for m in MODELS]
    ax.bar(xs + (width / 2 if EVAL_PRETRAINED_BASELINE else 0), fv, width, label='fine-tuned')
    ax.axhline(results[results.variant == 'noisy input'][col].iloc[0], color='gray', ls='--', label='noisy input')
    ax.set_xticks(xs, MODELS); ax.set_ylabel('Mean PSNR (dB)'); ax.set_title(title); ax.grid(axis='y', alpha=0.3)
    lo = min(results[col].dropna()); ax.set_ylim(max(0, lo - 3), max(results[col].dropna()) + 2)
axes[0][0].legend()
fig.suptitle(f'Test split ({len(test_rows)} pairs, {EVAL_PATCH}px RGB center crops)')
fig.tight_layout(); fig.savefig(TEST_DIR / 'test_comparison.png', dpi=160); plt.show()

# ---------- Sample images ----------
cols = ['Noisy'] + [m for m in MODELS] + ['Clean (GT)']
fig, axes = plt.subplots(len(vis_idx), len(cols), figsize=(3 * len(cols), 3 * len(vis_idx)), squeeze=False)
for r_i, idx in enumerate(vis_idx):
    imgs = [vis_noisy[r_i]] + [VIS_OUT[m][r_i] for m in MODELS] + [vis_clean[r_i]]
    for c_i, img in enumerate(imgs):
        axes[r_i][c_i].imshow(img.permute(1, 2, 0).clamp(0, 1).numpy()); axes[r_i][c_i].axis('off')
        if r_i == 0: axes[r_i][c_i].set_title(cols[c_i] + (' (fine-tuned)' if cols[c_i] in MODELS else ''))
    axes[r_i][0].text(0, -0.02, test_sources[idx], transform=axes[r_i][0].transAxes, va='bottom', fontsize=9)
fig.tight_layout(); fig.savefig(TEST_DIR / 'test_samples.png', dpi=150); plt.show()

## Stage 7. Export for submission and for the app

Copies the new `.pth` files, graphs, tables and logs into `submission_output/` (and a `.zip`). **To use the new models in your VS Code app, copy the three files in `submission_output/weights/` into the app's `weights/` folder** (they are already in `weights/` if you ran this notebook inside the project itself).

In [ ]:
for m in MODELS:
    shutil.copy2(ft_path(m), OUT / 'weights' / ft_path(m).name)
    CKPT_INFO[m]['sha256'] = sha256_file(OUT / 'weights' / ft_path(m).name)

manifest_info = {'created': datetime.datetime.now().isoformat(timespec='seconds'), 'manifest_sha256': manifest_sha256,
                 'training': {'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'patch': PATCH, 'batch_size': BATCH_SIZE,
                              'seed': SEED, 'loss': 'L1', 'validation_images': VALIDATION_LIMIT},
                 'evaluation': {'split': 'test', 'images': len(test_rows), 'crop': EVAL_PATCH},
                 'checkpoints': CKPT_INFO, 'environment': ENV}
(OUT / 'logs' / 'run_manifest.json').write_text(json.dumps(manifest_info, indent=2), encoding='utf8')
(OUT / 'HOW_TO_USE.txt').write_text(
    'Copy the .pth files in weights/ into the app weights/ folder (VS Code project):\n'
    + ''.join(f'  {ft_path(m).name}\n' for m in MODELS)
    + '\nOther folders: graphs/ (training curves), test_results/ (test-split tables and charts), logs/ (training logs, validation report, hashes).\n',
    encoding='utf8')

if EXPORT_ZIP:
    zip_path = shutil.make_archive(str(ROOT / 'submission_output'), 'zip', OUT)
    print('Zip:', zip_path)
    if IN_COLAB and DOWNLOAD_ZIP_IN_COLAB:
        try:
            from google.colab import files; files.download(zip_path)
        except Exception as exc:
            print('Automatic download unavailable:', exc)

print('\nExported files:')
for p in sorted(OUT.rglob('*')):
    if p.is_file(): print(f'  {p.relative_to(ROOT)}  ({p.stat().st_size / 1e6:.2f} MB)')
print('\nCopy these into the app weights folder:')
for m in MODELS: print('  ', OUT / 'weights' / ft_path(m).name)

## Reproducibility and limits

* One fixed manifest (SHA-256 recorded in `logs/run_manifest.json`) is shared by all three models; whole scenes stay inside a single split.
* Settings, seeds, checkpoint hashes and the software environment are saved in `submission_output/logs/`.
* Scene grouping prevents overlap among this project's splits but **does not remove overlap with the published SIDD pretraining** of Restormer and NAFNet; report SIDD and PolyU separately. DnCNN's published checkpoint was trained on Gaussian noise.
* Metrics are computed on small center crops (`EVAL_PATCH`), not full images, so absolute values are not directly comparable to full-image benchmark numbers.
* GPU-dependent training is not bit-for-bit reproducible across hardware, even with fixed seeds.